In [1]:
import pandas as pd

In [3]:
# Load the original data so the conflicting records are still present
raw_df = pd.read_excel("ADEI.xlsx")


In [4]:
# Standardise column names and check required fields
raw_df.columns = raw_df.columns.str.strip().str.lower()

required_columns = [
    "ticket_id", "opened_at", "channel", "category",
    "status", "resolved_at", "csat"
]

missing_columns = set(required_columns) - set(raw_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

df = raw_df[required_columns].copy()

In [5]:
# Clean text fields
for column in df.select_dtypes(include=["object", "string"]).columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

df["ticket_id"] = df["ticket_id"].str.upper()
df["status"] = df["status"].str.title()

# Convert timestamps
df["opened_at"] = pd.to_datetime(df["opened_at"], errors="coerce")
df["resolved_at"] = pd.to_datetime(df["resolved_at"], errors="coerce")

In [6]:
# Remove exact duplicate rows first
df = df.drop_duplicates(subset=required_columns, keep="first").copy()


In [7]:

# Identify IDs with multiple non-identical records
conflict_mask = (
    df["ticket_id"].notna()
    & df["ticket_id"].duplicated(keep=False)
)
conflicts = df.loc[conflict_mask].copy()

In [8]:
# Apply the same general exclusions under both rules:
# missing ticket ID or invalid/missing opening timestamp
eligible = df[
    df["ticket_id"].notna()
    & df["opened_at"].notna()
].copy()


In [9]:
# Rule 1: exclude every record with a conflicting ticket ID
rule_exclude = eligible[
    ~eligible["ticket_id"].isin(conflicts["ticket_id"])
].copy()

In [10]:
# Rule 2: retain the most complete record for each conflicting ID
conflicts["completeness"] = conflicts[required_columns].notna().sum(axis=1)

best_conflict_records = (
    conflicts
    .sort_values("completeness", ascending=False, kind="stable")
    .drop_duplicates(subset="ticket_id", keep="first")
    .drop(columns="completeness")
)

rule_keep_complete = pd.concat(
    [
        eligible[
            ~eligible["ticket_id"].isin(conflicts["ticket_id"])
        ],
        best_conflict_records
    ],
    ignore_index=True
)


In [11]:
# Calculate SLA results for each rule
SLA_HOURS = 24

def calculate_sla(data, rule_name):
    resolution_hours = (
        data["resolved_at"] - data["opened_at"]
    ).dt.total_seconds() / 3600

    valid_duration = (
        data["status"].eq("Resolved")
        & resolution_hours.notna()
        & resolution_hours.ge(0)
    )

    valid_sample = int(valid_duration.sum())
    breaches = int((resolution_hours[valid_duration] > SLA_HOURS).sum())

    return {
        "rule": rule_name,
        "ticket_count": len(data),
        "valid_resolution_sample": valid_sample,
        "sla_breaches": breaches,
        "sla_denominator": valid_sample,
        "sla_breach_rate_percent": (
            breaches / valid_sample * 100 if valid_sample else float("nan")
        ),
    }

comparison = pd.DataFrame([
    calculate_sla(rule_exclude, "Exclude all conflicting IDs"),
    calculate_sla(rule_keep_complete, "Keep most complete record"),
])

print("Conflicting ticket IDs:", conflicts["ticket_id"].unique().tolist())
print("Conflicting records:")
display(conflicts[required_columns])

print("SLA comparison (24-hour target)")
display(comparison.round(2))

Conflicting ticket IDs: ['TK-1111']
Conflicting records:


,ticket_id,opened_at,channel,category,status,resolved_at,csat
111,TK-1111,2026-10-29 09:06:00,WhatsApp,Access,Resolved,2026-10-29 10:55:00,NaN
145,TK-1111,2026-10-29 09:06:00,WhatsApp,Access,Open,NaT,NaN


SLA comparison (24-hour target)


,rule,ticket_count,valid_resolution_sample,sla_breaches,sla_denominator,sla_breach_rate_percent
0,Exclude all conflicting IDs,139,94,1,94,1.06
1,Keep most complete record,140,95,1,95,1.05


- Exclude all conflicting IDs: 139 tickets; 1 breach out of 94 valid resolutions (1.06%).
- Keep the most complete record: 140 tickets; 1 breach out of 95 valid resolutions (1.05%).
  
The conflicting ID is TK-1111. The more complete record is marked Resolved and took about 1.82 hours, so it adds one valid resolution without adding a breach. The recommendation does not materially change: investigate the single SMS breach and monitor performance, while treating the SMS rate cautiously because it is based on one ticket. For your primary results, excluding the conflict until the source record is verified is the more conservative choice; report the “keep most complete” result as a sensitivity comparison.